# 1. Generating and running benchmarking circuits

In this notebook we generate random benchmarking parameters, the corresponding (Clifford) benchmarking circuits and then transpile and run these circuits on the `ibm_brisbane` device. 

In [1]:
# Initial imports:

from IPython.display import clear_output
import sys
sys.path.append('../')
import time
import numpy as np

# Our custom functions
print('Importing fccb...')
import fccb

clear_output(wait=True)
print('Importing Qiskit bits...')

# Qiskit functions and estimator primitives
from qiskit import QuantumCircuit, qasm3, transpile
from qiskit.transpiler.passes import RemoveBarriers
from qiskit.primitives import Estimator as Estimator_sim
from qiskit.circuit.library import UnitaryGate
from qiskit_aer.primitives import Estimator as Estimator_Aer
estimator_clifford = Estimator_Aer(backend_options=dict(method="stabilizer"))
estimator_exact = Estimator_sim()

# Qiskit Runtime
clear_output(wait=True)
print('Initialising QiskitRuntimeService...')
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2, Batch
service = QiskitRuntimeService(name='MY-ACCOUNT-HERE')

# Setup a backend and import its properties
clear_output(wait=True)
print('Importing backend...')
hardware = service.get_backend('ibm_brisbane')
basis_gates = hardware.basis_gates
coupling_map = hardware.coupling_map

# Three circuit layers required to implement 2-qubit interactions
Ls = np.load('../layers.npy')

clear_output(wait=True)
print('Done!')

Done!


We first define a function to generate our Clifford benchmarking circuits, given the total number of qubits in the circuit $N$, a list of specific qubits in the hardware to use and the number of (single + two-qubit) layers in the circuit.

In [2]:
# Dictionary showing how Pauli matrices with theta = pi/2 map between X/Y/Z +/- eigenstates
rot_dict = {'X':{'X+':'X+','X-':'X-','Y+':'Z+','Y-':'Z-','Z+':'Y-','Z-':'Y+'},
            'Y':{'X+':'Z-','X-':'Z+','Y+':'Y+','Y-':'Y-','Z+':'X+','Z-':'X-'},
            'Z':{'X+':'Y+','X-':'Y-','Y+':'X-','Y-':'X+','Z+':'Z+','Z-':'Z-'}}

def circuit_cliff_evolve_with_structure(N, device_qubits, n_layers):
    
    # Initial state of all qubits is |0>
    state_dict = {i:'Z+' for i in device_qubits}
    gates = []
    
    # For each circuit layer:
    for layer in range(n_layers):
        
        # Apply random single-qubit Pauli rotation to each qubit; update state
        for i in device_qubits:
            P = np.random.choice(['X','Y','Z'])
            gates.append([P,[i]])
            state_dict[i] = rot_dict[P][state_dict[i]]

        # For each of the 3 two-qubit interaction layers, consider all pairs of qubits
        # from that layer which are also in our circuit
        for Li in Ls:
            len_L = len(Li)
            for i in range(len_L):
                q0,q1 = Li[i]
                if q0 in device_qubits and q1 in device_qubits:
                    
                    # Apply Pauli P = P0 x P1 
                    # Pick P0 such that q0 an eigenstate, pick P1 at random
                    P0 = state_dict[q0][0]
                    P1 = np.random.choice(['X','Y','Z'])
                    P = P0 + P1
                    
                    # State q0 does not change; state q1 gets rotated in Bloch sphere
                    state_dict[q1] = rot_dict[P1][state_dict[q1]]

                    gates.append([P,[q0,q1]])
    
    
    # Amend circuit using one single-qubit Pauli rotation (up to concatenation) 
    # so that <Z62>=1
    state_62 = state_dict[62]
    
    if state_62 == 'X+':
        gates.append(['Y',[62]])
        gates.append(['Y',[62]])
        gates.append(['Y',[62]])
    elif state_62 == 'X-':
        gates.append(['Y',[62]])
    elif state_62 == 'Y+':
        gates.append(['X',[62]])
    elif state_62 == 'Y-':
        gates.append(['X',[62]])
        gates.append(['X',[62]])
        gates.append(['X',[62]])
    elif state_62 == 'Z-':
        gates.append(['X',[62]])
        gates.append(['X',[62]])
           
    return gates

Now we define a function to, given a device coupling map and a fixed number of qubits $N$, generate a set of benchmarking circuit parameters: pick a connected subset of $N$ device qubits containing qubit 62 and generate the corresponding benchmarking circuit corresponding to the measurement of $\langle Z_{62}\rangle$. The circuit data returns as a list of gates where each gate is something like `['XY', [q0, q1]]`, in this case indicating the gate $e^{iX_{q_0}Y_{q_1}\theta/2}$.

In [3]:
def generate_experiment(coupling_map, N, n_layers):

    # Pick random connected set of N device qubits to use, starting at qubit 62 
    central_qubit = 62
    device_qubits = list([central_qubit])
    for i in range(N-1):
        connected_qubits_list = []
        for q in device_qubits:
            connected_qubits_list = connected_qubits_list + fccb.connected_qubits(q, coupling_map)
        connected_qubits_list = [q for q in connected_qubits_list if q not in device_qubits]
        new_qubit = np.random.choice(connected_qubits_list)
        device_qubits.append(new_qubit)

    device_qubits.sort()

    # Generate length-N Pauli string to measure, 
    # where first element in string acts on first qubit in device_qubits and so on
    i = device_qubits.index(central_qubit)
    pauli_to_measure = 'I'*i + 'Z' + 'I'*(N-i-1)

    # Generate corresponding benchmarking circuit
    circuit_gates = circuit_cliff_evolve_with_structure(N, device_qubits, n_layers)

    clear_output(wait=True)
    return device_qubits, pauli_to_measure, circuit_gates

We include a couple of functions to allow us to compile circuits that can be passed to the Clifford simulator. 

In [4]:
H_Y_unitary = (1/np.sqrt(2))*np.matrix([[1, -1j],[1j,-1]])
H_Y = UnitaryGate(H_Y_unitary, label='Hy')

H_X_unitary = (1/np.sqrt(2))*np.matrix([[1, 1],[1,-1]])
H_X = UnitaryGate(H_X_unitary, label='H')

H_Z_unitary = np.matrix([[1, 0],[0,1]])
H_Z = UnitaryGate(H_Z_unitary, label='I')

def clifford_pauli_rotation(P, theta): # Circ for exp(-iP*theta)
    
    # Note qubit order is reversed compared to intuition due to Qiskit qubit ordering
    
    qc = QuantumCircuit(2)
    
    P1, P2 = P
    H_P = {'X': H_X, 'Y': H_Y, 'Z': H_Z}
    
    qc.append(H_P[P1], [1])
    qc.append(H_P[P2], [0])
    
    qc.rzz(2*theta,0,1)
    
    qc.append(H_P[P1], [1])
    qc.append(H_P[P2], [0])
    
    return qc

In [5]:
def pauli_list_to_clifford_circuit(gates_list, device_qubits):

    N = len(device_qubits)
    qc = QuantumCircuit(N)
    
    for i in range(len(gates_list)):
        P, Qs = gates_list[i]
        Qs_small = [device_qubits.index(q) for q in Qs]
        
        if len(P) == 1:
            if P == 'X':
                qc.rx(np.pi/2, Qs_small[0])
            elif P == 'Y':
                qc.ry(np.pi/2, Qs_small[0])
            elif P == 'Z':
                qc.rz(np.pi/2, Qs_small[0])       
        else:
            qc_P_rot = clifford_pauli_rotation(P[::-1], np.pi/4)
            qc = qc.compose(qc_P_rot, qubits=Qs_small)
      
    return qc

Quick test that we can generate a benchmarking circuit, compile it and run it on the Clifford simulator to measure $\langle Z_{62} \rangle = 1$:

In [6]:
N = 127
n_layers = 2
print(N, 'qubits, ', n_layers, 'layers')

127 qubits,  2 layers


In [7]:
device_qubits, pauli_to_measure, gates_list = generate_experiment(coupling_map, N, n_layers)

In [8]:
qc = pauli_list_to_clifford_circuit(gates_list, device_qubits)
qc.draw()

┌─────────┐┌────┐            ┌────┐    ┌────┐                       »
  q_0: ┤ Ry(π/2) ├┤ Hy ├─■──────────┤ Hy ├────┤ Hy ├──────────────■────────»
       ├─────────┤├───┬┘ │          └────┘    ├───┬┘     ┌────┐   │ZZ(π/2) »
  q_1: ┤ Ry(π/2) ├┤ I ├──┼─────────■──────────┤ I ├──────┤ Hy ├───■────────»
       ├─────────┤├───┤  │         │ZZ(π/2)   ├───┤      ├───┬┘            »
  q_2: ┤ Ry(π/2) ├┤ H ├──┼─────────■──────────┤ H ├──────┤ H ├─────────────»
       ├─────────┤├───┤  │                    └───┘      └───┘             »
  q_3: ┤ Ry(π/2) ├┤ H ├──┼────────────────────────────────────────■────────»
       ├─────────┤├───┤  │                    ┌───┐      ┌───┐    │ZZ(π/2) »
  q_4: ┤ Ry(π/2) ├┤ H ├──┼─────────■──────────┤ H ├──────┤ H ├────■────────»
       ├─────────┤├───┤  │         │ZZ(π/2)   ├───┤      ├───┤             »
  q_5: ┤ Rz(π/2) ├┤ H ├──┼─────────■──────────┤ H ├──────┤ I ├────■────────»
       ├─────────┤├───┤  │                    ├───┤      ├───┤    │ZZ(π/2) »
  q_6: ┤ Ry(π/2) ├┤ H ├──┼─────────■──────────┤ H ├──────┤ H ├────■────────»
       ├─────────┤├───┴┐ │         │ZZ(π/2)   ├───┴┐     ├───┤             »
  q_7: ┤ Ry(π/2) ├┤ Hy ├─┼─────────■──────────┤ Hy ├─────┤ I ├────■────────»
       ├─────────┤├───┬┘ │                    ├───┬┘     ├───┤    │ZZ(π/2) »
  q_8: ┤ Rz(π/2) ├┤ I ├──┼─────────■──────────┤ I ├──────┤ H ├────■────────»
       ├─────────┤├───┤  │         │ZZ(π/2)   ├───┤      ├───┤             »
  q_9: ┤ Rz(π/2) ├┤ I ├──┼─────────■──────────┤ I ├──────┤ H ├────■────────»
       ├─────────┤├───┴┐ │                    ├───┴┐     ├───┴┐   │ZZ(π/2) »
 q_10: ┤ Rx(π/2) ├┤ Hy ├─┼─────────■──────────┤ Hy ├─────┤ Hy ├───■────────»
       ├─────────┤├───┬┘ │         │ZZ(π/2)   ├───┬┘     ├────┤            »
 q_11: ┤ Rz(π/2) ├┤ H ├──┼─────────■──────────┤ H ├──────┤ Hy ├────────────»
       ├─────────┤├───┤  │                    ├───┤      ├───┬┘            »
 q_12: ┤ Ry(π/2) ├┤ H ├──┼─────────■──────────┤ H ├──────┤ H ├─────────────»
       ├─────────┤├───┤  │         │ZZ(π/2)   ├───┤   ┌──┴───┴──┐  ┌───┐   »
 q_13: ┤ Rz(π/2) ├┤ I ├──┼─────────■──────────┤ I ├───┤ Rz(π/2) ├──┤ I ├───»
       ├─────────┤├───┴┐ │ZZ(π/2)   ┌────┐    ├───┴┐  └─────────┘  └───┘   »
 q_14: ┤ Rx(π/2) ├┤ Hy ├─■──────────┤ Hy ├────┤ Hy ├───────────────────────»
       ├─────────┤├───┬┘            └────┘    ├───┬┘     ┌───┐             »
 q_15: ┤ Rz(π/2) ├┤ I ├────────────■──────────┤ I ├──────┤ H ├─────────────»
       ├─────────┤├───┴┐           │          └───┘      └───┘             »
 q_16: ┤ Rx(π/2) ├┤ Hy ├───────────┼───────────────────────────────────────»
       ├─────────┤├───┬┘           │                     ┌───┐     ┌───┐   »
 q_17: ┤ Ry(π/2) ├┤ H ├────────────┼─────────■───────────┤ H ├─────┤ H ├───»
       ├─────────┤├───┤            │         │           ├───┤     ├───┤   »
 q_18: ┤ Ry(π/2) ├┤ H ├──■─────────┼─────────┼───────────┤ H ├─────┤ H ├───»
       ├─────────┤├───┤  │ZZ(π/2)  │         │           ├───┤     ├───┴┐  »
 q_19: ┤ Rx(π/2) ├┤ H ├──■─────────┼─────────┼───────────┤ H ├─────┤ Hy ├──»
       ├─────────┤├───┴┐           │         │           ├───┴┐    ├────┤  »
 q_20: ┤ Rx(π/2) ├┤ Hy ├─■─────────┼─────────┼───────────┤ Hy ├────┤ Hy ├──»
       ├─────────┤├────┤ │ZZ(π/2)  │         │           ├────┤    ├────┤  »
 q_21: ┤ Rx(π/2) ├┤ Hy ├─■─────────┼─────────┼───────────┤ Hy ├────┤ Hy ├──»
       ├─────────┤├───┬┘           │ZZ(π/2)  │           ├───┬┘    ├───┬┘  »
 q_22: ┤ Ry(π/2) ├┤ H ├────────────■─────────┼───────────┤ H ├─────┤ H ├───»
       ├─────────┤├───┤             ┌───┐    │           ├───┤     └───┘   »
 q_23: ┤ Rx(π/2) ├┤ I ├──■──────────┤ I ├────┼───────────┤ H ├─────────────»
       ├─────────┤├───┤  │ZZ(π/2)   ├───┤    │           ├───┤             »
 q_24: ┤ Ry(π/2) ├┤ H ├──■──────────┤ H ├────┼───────────┤ I ├────■────────»
       ├─────────┤├───┴┐            ├───┴┐   │           ├───┤    │ZZ(π/2) »
 q_25: ┤ Rz(π/2) ├┤ Hy ├─■──────────┤ Hy ├───┼───────────┤ H ├────■────────»
   

In [9]:
# Note we have to reverse the Pauli string to respect Qiskit's qubit ordering

# (takes a little while)
estimator_clifford.run(qc, pauli_to_measure[::-1]).result().values[0] 

1.0

The next step is to generate a list of benchmarking circuit parameters. We will have 7 different values of $N$ (2,4,8,16,32,64,127), 15 possible values for the number of circuit layers $\text{n_layers}$ (1,2,...,15), and 10 repetitions for each ($N$, $\text{n_layers}$) combination.

In [10]:
experiment_data_3D_array = [[[0]*10 for i in range(15)] for j in range(7)]

n_qubits_array = [2,4,8,16,32,64,127]
n_layers_array = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]

In [11]:
# # Generate experimental parameters

# for int_qubits in range(len(n_qubits_array)):
#     for int_layers in range(len(n_layers_array)):
        
#         n_qubits = n_qubits_array[int_qubits]
#         n_layers = n_layers_array[int_layers]
        
#         # 10 experiments per (n_qubits, n_layers) pair
#         for int_experiment_rep in range(10):
            
#             if experiment_data_3D_array[int_qubits][int_layers][int_experiment_rep] == 0:
                
#                 # Generate experiment and circuit
#                 device_qubits, pauli_to_measure, circuit_gates = generate_experiment(coupling_map, n_qubits, n_layers)

#                 # Save parameters
#                 experiment_data_3D_array[int_qubits][int_layers][int_experiment_rep] = [device_qubits, pauli_to_measure, circuit_gates]
                
#                 clear_output(wait=True)
#                 print('Done for ', int_qubits, int_layers, int_experiment_rep)

In [12]:
# np.save('benchmarking_parameters.npy', experiment_data_3D_array)

In [13]:
experiment_data_3D_array = np.load('benchmarking_parameters.npy', allow_pickle=True)

In [14]:
# Quick test
device_qubits, pauli_to_measure, circuit_gates = experiment_data_3D_array[2][4][1]
qc = pauli_list_to_clifford_circuit(circuit_gates, device_qubits)
estimator_clifford.run(qc, pauli_to_measure[::-1]).result().values[0]

1.0

The next step is to compile the circuits to be run on IBM hardware. We first use a custom transpiling function which transpiles the circuit using the basis gate set of the device, before using the Qiskit transpiler for basic circuit optimisation, where adjacent $R_Z$ rotations are combined and the circuit barriers are ignored.

In [15]:
observables_list = ['I'*64 + 'Z' + 'I'*62]*1050

In [16]:
# circuits_list = []

# rb = RemoveBarriers()

# # Loop over experiment parameters
# for int_qubits in range(7):
#     for int_layers in range(15):
        
#         n_qubits = n_qubits_array[int_qubits]
#         n_random_layers = n_layers_array[int_layers]
        
#         for int_experiment_rep in range(10):

#             # Load data for each experiment
#             device_qubits, pauli_to_measure, circuit_gates = experiment_data_3D_array[int_qubits][int_layers][int_experiment_rep]

#             # Custom transpilation
#             qc = fccb.pauli_list_to_circuit(circuit_gates, device_qubits,transpiled=True, full_device=True, basis_gates=basis_gates, 
#                                             coupling_map=coupling_map, theta=np.pi/4)
#             # Qiskit transpilation
#             qc = transpile(rb(qc), hardware)
#             # Save
#             circuits_list.append(qc)

#             clear_output(wait=True)
#             print('Done for ', int_qubits, int_layers, int_experiment_rep)

We save the transpiled circuits as a series of `.qasm` files in their own folder. For each circuit, we add an empty $R_Z(0)$ rotation to qubit 126 so that the circuits remain 127-qubit circuits when we load them later. 

In [17]:
# # Save transpiled circuits

# from qiskit.qasm3 import dump

# for i in range(1050):
    
#     filename = 'transpiled_benchmarking_circuits/q' + str(i) + '.qasm'
    
#     with open(filename, 'w') as stream:
#         dump(circuits_list_trans[i], stream)

#     with open(filename, 'a') as file:
#         file.write('rz(0) $126;')

In [18]:
# load transpiled circuits into QuantumCircuit objects

from qiskit.qasm3 import load

circuits_list_trans = []
for i in range(1050):
    filename = 'transpiled_benchmarking_circuits/q' + str(i) + '.qasm'
    circuits_list_trans.append(load(filename))
    clear_output(wait=True)
    print(i)

1049


The following partitioning allows us to run our benchmarking circuits using the Qiskit open plan. Each job may contain a maximum of 300 circuits, and one may only queue up to 3 jobs at once. Hence, we submit the first 900 circuits as 3 jobs and submit the remaining circuits as a 4th job later. 

In [19]:
circuits_list_of_lists = [0]*4
circuits_list_of_lists[0] = circuits_list_trans[0:300]
circuits_list_of_lists[1] = circuits_list_trans[300:600]
circuits_list_of_lists[2] = circuits_list_trans[600:900]
circuits_list_of_lists[3] = circuits_list_trans[900:]

observables_list_of_lists = [0]*4
observables_list_of_lists[0] = observables_list[0:300]
observables_list_of_lists[1] = observables_list[300:600]
observables_list_of_lists[2] = observables_list[600:900]
observables_list_of_lists[3] = observables_list[900:]

In [20]:
# # Submit data to IBM backend as 4 separate jobs
# # (can only queue up to 3 at once)

# job_ids = []

# with Batch(service=service, backend=hardware) as batch:
        
#         # 1024 shots per circuit; no QEM
#         estimator_noisy = EstimatorV2(batch)
#         estimator_noisy.options.default_shots = 1024
#         estimator_noisy.options.resilience_level = 0
        
#         for i in [0,1,2]:
#             tuples = [(circuits_list_of_lists[i][j], observables_list_of_lists[i][j]) for j in range(len(circuits_list_of_lists[i]))]
#             job = estimator_noisy.run(tuples)
#             job_id = job.job_id()
#             job_ids.append(job_id)
#             print('Job ID ' + str(i+1) + ': ', job_id)

In [21]:
# with Batch(service=service, backend=hardware) as batch:
        
#         # 1024 shots per circuit; no QEM
#         estimator_noisy = EstimatorV2(batch)
#         estimator_noisy.options.default_shots = 1024
#         estimator_noisy.options.resilience_level = 0
        
#         for i in [3]:
#             tuples = [(circuits_list_of_lists[i][j], observables_list_of_lists[i][j]) for j in range(len(circuits_list_of_lists[i]))]
#             job = estimator_noisy.run(tuples)
#             job_id = job.job_id()
#             job_ids.append(job_id)
#             print('Job ID ' + str(i+1) + ': ', job_id)

Once jobs are done, import and save the raw results.

In [22]:
job_ids = ['','','','']

In [23]:
job_values = []
for i in range(4):
    vals = [float(res.data.evs) for res in service.job(job_id).result()]
    for val in vals:
        job_values.append(val)
    print('Job', str(i+1), '- results imported')

Job 1 - results imported
Job 2 - results imported
Job 3 - results imported
Job 4 - results imported


In [24]:
# Ensure all values are between 0 and 1
for i in range(len(job_values)):
    if job_values[i] < 0:
        job_values[i] = 0
    elif job_values[i] > 1:
        job_values[i] = 1

In [25]:
np.save('benchmarking_job_data_brisbane.npy',job_values)

We process and plot this data in Notebook 3. 